# Camada Gold

Camada final com objetivo criar agregações e métricas para responder as perguntas:

1) Existem diferenças relevantes de desempenho entre escolas públicas e privadas?
2) Qual a diferença de desempenho das escolas de diferentes niveis socioeconômicos?
3) Quais estados apresentam os melhores e piores desempenhos médios no SAEB?
4) Quais estados entregam um desempenho educacional abaixo do esperado para seu nível de desenvolvimento?
5) Existem estados que se destacam por apresentar bons resultados educacionais mesmo possuindo indicadores socioeconômicos inferiores?

# 1.0 Preparação 



## 1.1. Importação de bibliotecas e bases 
Importação das bibliotecas necessárias e leitura das bases a serem utilizadas provenientes das camadas silver e bronze

In [0]:
#Importação de bases 
from pyspark.sql.functions import when, col
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
#Leitura das 3 bases a serem utilizadas para gerar as métricas e agregações necessárias
df_saeb = spark.table("`mvp-eng-dados-puc-rio`.silver.resultados_saeb") #Base da camada silver tratada com resultados do SAEB
df_atributos_estados = spark.table("`mvp-eng-dados-puc-rio`.bronze.atributos_estados") # Base dimensão da camada bronze com a lista de estados e suas respectivas regiões
df_indicadores_estados = spark.table("`mvp-eng-dados-puc-rio`.bronze.dados_socioeconomicos_estados") #Base de fatos da camada bronze com indicadores socioeconômicos dos estados para diferentes censos do IBGE


In [0]:
#Seleção da camada gold para salvar as bases geradas

spark.sql("USE CATALOG `mvp-eng-dados-puc-rio`")
spark.sql("USE SCHEMA `gold`")


DataFrame[]

In [0]:
#visualização de amostra da base df_saeb
display(df_saeb.limit(10))

ID_SAEB,ID_MUNICIPIO,ID_ESCOLA,PC_FORMACAO_DOCENTE_INICIAL,PC_FORMACAO_DOCENTE_FINAL,PC_FORMACAO_DOCENTE_MEDIO,NIVEL_SOCIO_ECONOMICO,NU_MATRICULADOS_CENSO_5EF,NU_PRESENTES_5EF,TAXA_PARTICIPACAO_5EF,NIVEL_0_LP5,NIVEL_1_LP5,NIVEL_2_LP5,NIVEL_3_LP5,NIVEL_4_LP5,NIVEL_5_LP5,NIVEL_6_LP5,NIVEL_7_LP5,NIVEL_8_LP5,NIVEL_9_LP5,NIVEL_0_MT5,NIVEL_1_MT5,NIVEL_2_MT5,NIVEL_3_MT5,NIVEL_4_MT5,NIVEL_5_MT5,NIVEL_6_MT5,NIVEL_7_MT5,NIVEL_8_MT5,NIVEL_9_MT5,NIVEL_10_MT5,NU_MATRICULADOS_CENSO_9EF,NU_PRESENTES_9EF,TAXA_PARTICIPACAO_9EF,NIVEL_0_LP9,NIVEL_1_LP9,NIVEL_2_LP9,NIVEL_3_LP9,NIVEL_4_LP9,NIVEL_5_LP9,NIVEL_6_LP9,NIVEL_7_LP9,NIVEL_8_LP9,NIVEL_0_MT9,NIVEL_1_MT9,NIVEL_2_MT9,NIVEL_3_MT9,NIVEL_4_MT9,NIVEL_5_MT9,NIVEL_6_MT9,NIVEL_7_MT9,NIVEL_8_MT9,NIVEL_9_MT9,NU_MATRICULADOS_CENSO_EM,NU_PRESENTES_EM,TAXA_PARTICIPACAO_EM,NIVEL_0_LPEM,NIVEL_1_LPEM,NIVEL_2_LPEM,NIVEL_3_LPEM,NIVEL_4_LPEM,NIVEL_5_LPEM,NIVEL_6_LPEM,NIVEL_7_LPEM,NIVEL_8_LPEM,NIVEL_0_MTEM,NIVEL_1_MTEM,NIVEL_2_MTEM,NIVEL_3_MTEM,NIVEL_4_MTEM,NIVEL_5_MTEM,NIVEL_6_MTEM,NIVEL_7_MTEM,NIVEL_8_MTEM,NIVEL_9_MTEM,NIVEL_10_MTEM,MEDIA_5EF_LP,MEDIA_5EF_MT,MEDIA_9EF_LP,MEDIA_9EF_MT,MEDIA_EM_LP,MEDIA_EM_MT,ESTADO,area,escola_publica,localizacao,Escola_Avaliada_5EF,Escola_Avaliada_9EF,Escola_Avaliada_EM,PROFICIENCIA_5EF_LP,PROFICIENCIA_5EF_MT,PROFICIENCIA_9EF_LP,PROFICIENCIA_9EF_MT,PROFICIENCIA_EM_LP,PROFICIENCIA_EM_MT
2023,6322170,61400934,100.0,43.2,null,N�vel V,14,15,107.14,"13,33",20.0,"26,67","13,33","13,33","13,33",0.0,0.0,0.0,0.0,6.67,26.67,20.0,13.33,20.0,13.33,0.0,0.0,0.0,0.0,0.0,15,10,66.67,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,166.45,175.9,null,null,null,null,Rondônia,Interior,Pública,Urbana,Sim,Não,Não,Básico,Básico,Sem resultado,Sem resultado,Sem resultado,Sem resultado
2023,6322170,61403177,100.0,55.6,null,N�vel IV,19,19,100.0,5.26,31.58,26.32,26.32,5.26,5.26,0.0,0.0,0.0,0.0,15.79,31.58,21.05,15.79,5.26,0.0,5.26,5.26,0.0,0.0,0.0,30,25,83.33,12.0,40.0,4.0,28.0,16.0,0.0,0.0,0.0,0.0,8.0,32.0,20.0,20.0,12.0,4.0,4.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,163.69,162.94,235.83,240.96,null,null,Rondônia,Interior,Pública,Urbana,Sim,Sim,Não,Básico,Abaixo do Básico,Básico,Básico,Sem resultado,Sem resultado
2023,6322170,61412274,56.3,85.2,91.1,N�vel IV,31,31,100.0,0,12.9,9.68,12.9,9.68,19.35,22.58,6.45,6.45,0.0,0.0,3.23,3.23,6.45,19.35,25.81,22.58,9.68,6.45,3.23,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,111,111,100.0,32.41,14.87,20.06,14.35,11.99,4.52,1.8,0.0,0.0,26.09,21.91,22.31,13.68,13.39,0.9,0.9,0.82,0.0,0.0,0.0,222.33,244.1,null,null,253.38,254.41,Rondônia,Interior,Pública,Urbana,Sim,Não,Sim,Adequado,Adequado,Sem resultado,Sem resultado,Básico,Abaixo do Básico
2023,6322170,61416961,100.0,45.5,null,N�vel IV,18,13,72.22,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,12,11,91.67,18.18,18.18,0.0,18.18,27.27,18.18,0.0,0.0,0.0,9.09,9.09,36.36,18.18,27.27,0.0,0.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,257.79,249.24,null,null,Rondônia,Interior,Pública,Rural,Não,Sim,Não,Sem resultado,Sem resultado,Básico,Básico,Sem resultado,Sem resultado
2023,6322170,61420915,null,73.2,70.3,N�vel IV,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,111,106,95.5,12.42,17.84,21.69,16.89,16.13,9.44,5.59,0.0,0.0,11.34,14.16,18.1,22.47,11.25,11.41,9.4,1.86,0.0,0.0,66,61,92.42,11.06,6.42,14.9,29.08,16.6,12.59,9.36,0.0,0.0,8.38,11.41,17.93,27.66,19.63,10.26,3.13,1.61,0.0,0.0,0.0,null,null,249.09,259.25,288.04,285.14,Rondônia,Interior,Pública,Urbana,Não,Sim,Sim,Sem resultado,Sem resulta

In [0]:
#visualização de amostra da base df_atributos_estados

display(df_atributos_estados.limit(10))

Estado,Região
Distrito Federal,Centro-Oeste
Goiás,Centro-Oeste
Mato Grosso,Centro-Oeste
Mato Grosso do Sul,Centro-Oeste
Alagoas,Nordeste
Bahia,Nordeste
Ceará,Nordeste
Maranhão,Nordeste
Paraíba,Nordeste
Pernambuco,Nordeste


In [0]:
#visualização de amostra da base df_indicadores_estados

display(df_indicadores_estados.limit(10))

ANO,UFN,E_ANOSESTUDO,GINI,RDPC,IDHM,IDHM_E,IDHM_L,IDHM_R
1991,Rondônia,7.5500000000000000,0.62000000000000000,304.89999999999998,0.40699999999999997,0.18099999999999999,0.63500000000000001,0.58499999999999996
1991,Acre,6.5600000000000000,0.63000000000000000,284.95999999999998,0.40200000000000002,0.17599999999999999,0.64500000000000002,0.57399999999999995
1991,Amazonas,6.5200000000000000,0.62000000000000000,345.82000000000000,0.43000000000000000,0.20399999999999999,0.64500000000000002,0.60499999999999998
1991,Roraima,7.1400000000000000,0.63000000000000000,437.24000000000000,0.45900000000000002,0.24000000000000000,0.62800000000000000,0.64300000000000002
1991,Pará,6.4800000000000000,0.62000000000000000,273.22000000000003,0.41299999999999998,0.19400000000000001,0.64000000000000000,0.56699999999999995
1991,Amapá,7.8100000000000000,0.57999999999999996,378.57000000000000,0.47199999999999998,0.25400000000000000,0.66800000000000004,0.62000000000000000
1991,Tocantins,6.3600000000000000,0.63000000000000000,243.58000000000000,0.36899999999999999,0.15500000000000000,0.58899999999999997,0.54900000000000004
1991,Maranhão,6.2900000000000000,0.60000000000000000,156.47000000000000,0.35699999999999998,0.17299999999999999,0.55100000000000005,0.47799999999999998
1991,Piauí,5.8900000000000000,0.64000000000000000,167.03000000000000,0.36199999999999999,0.16400000000000001,0.59499999999999997,0.48799999999999999
1991,Ceará,6.2700000000000000,0.66000000000000000,219.83000000000000,0.40500000000000003,0.20399999999999999,0.61299999999999999,0.53200000000000003


## 1.2. Filtragem das tabelas

Com o objetivo de preparar as tabelas para análise, alguns filtros serão feitos nas bases da camada silver e bronze utilizadas




### 1.2.1. Filtragem da base df_saeb

Foi definido para a análise a avaliação dos dados do último SAEB para o 9º ano do ensino fundamental, para isso foram feitos os filtros:

1) Filtrar a base com o objetivo de utilizar somente o último SAEB para geração das tabelas da camada gold. Na base original só há o resultado do SAEB de 2023, mas esse filtro deixará a camada gold preparada para analisar o resultado de novos SAEBs, caso seja feito o upload desses dados
2) Filtrar somente escolas que foram avaliadas no 9º ano do ensino fundamental

In [0]:
1# 1 e 2) Filtrar escolas avaliadas no 9º EF e último SAEB
# ============================================================

# Obtenção dos registros do último SAEB realizando um filtro na coluna ID_SAEB a partir do seu maior valor com resultados para o 9º ano do ensino fundamental
max_id_saeb = (
    df_saeb
    .filter(F.col("Escola_Avaliada_9EF") == "Sim")
    .agg(F.max("ID_SAEB").alias("max_id_saeb"))
    .collect()[0]["max_id_saeb"]
)

df_saeb_filtrado  = (
    df_saeb
    .filter(
        (F.col("Escola_Avaliada_9EF") == "Sim") &
        (F.col("ID_SAEB") == max_id_saeb)
    )
)


In [0]:
#célula criada para avaliar se o filtro realizado funcionou

qtd_antes = df_saeb.count()
qtd_depois = df_saeb_filtrado.count()
print(f"quantidade antes: {qtd_antes}")
print(f"quantidade depois: {qtd_depois}")


quantidade antes: 70151
quantidade depois: 31080


Como pode ser visto, nossa base diminuiu, mantendo somente as escolas avaliadas no 9º ano do ensino fundamental

### 1.2.2. Filtragem da base de indicadores socioeconômicos

Nesta base foi feito um filtro para somente pegar os indicadores do último censo do IBGE disponível


In [0]:
# ============================================================
#Filtrar df_indicadores_estados pelo maior ANO de realização do censo do IBGE
# ============================================================

max_ano = (
    df_indicadores_estados
    .agg(F.max("ANO").alias("max_ano"))
    .collect()[0]["max_ano"]
)

df_indicadores_filtrado = (
    df_indicadores_estados
    .filter(F.col("ANO") == max_ano)
)

In [0]:
#célula criada para avaliar se o filtro realizado funcionou

qtd_antes = df_indicadores_estados.count()
qtd_depois = df_indicadores_filtrado.count()
print(f"quantidade antes: {qtd_antes}")
print(f"quantidade depois: {qtd_depois}")
df_indicadores_filtrado

quantidade antes: 81
quantidade depois: 27


DataFrame[ANO: bigint, UFN: string, E_ANOSESTUDO: decimal(36,16), GINI: decimal(17,17), RDPC: decimal(34,14), IDHM: decimal(17,17), IDHM_E: decimal(17,17), IDHM_L: decimal(17,17), IDHM_R: decimal(17,17)]

Como pode ser visto acima sobraram 27 registros (1 por estado) mostrando que a filtragem funcionou

# 2. Geração de tabela para responder se existem diferenças relevantes de desempenho entre escolas públicas e privadas

Para isso, será feita uma tabela que faça o agrupamento por tipo de escola e calcule as médias de português e matemática

In [0]:
#Agrupamento pelo tipo de escola (escola_publica), contagem da quantidade de escola por tipo e média das notas de português e matemática

df_resumo_publico_privada = (
    df_saeb_filtrado
    .groupBy("escola_publica")
    .agg(
        F.count("*").alias("quantidade_escolas"),
        F.avg("MEDIA_9EF_LP").alias("media_9EF_LP"),
        F.avg("MEDIA_9EF_MT").alias("media_9EF_MT")
    )
    .orderBy("escola_publica")
)

display(df_resumo_publico_privada)

escola_publica,quantidade_escolas,media_9EF_LP,media_9EF_MT
Pública,31080,251.86352155727243,250.14831628056683


Como só há escolas públicas, não é possível avaliar o resultado entre públicas e privadas, entretanto, se no futuro forem adicionadas, será gerada uma tabela com essa informação

## 2.1. Salvar base na camada gold

In [0]:
df_resumo_publico_privada.write.format("delta").mode("overwrite").saveAsTable("resultado_saeb_por_tipo_de_escola")

# 3. Geração de tabela por nível socioeconômico para responder a pergunta 2
(Qual a diferença de desempenho das escolas de diferentes niveis socioeconômicos?)

Para responder essa pergunta, é preciso explicar as classificações do nível socioeconômico

- **Nível I** : O estrato mais baixo; famílias com menor escolaridade (até o fundamental incompleto) e posse restrita de bens básicos (como geladeira, TV e celular).
- **Níveis II, III e IV**: Indicam faixas de transição com aumento gradual de itens de conforto em casa e anos de estudo dos responsáveis.
- **Níveis V e VI**: Concentram a maior parte das escolas e estudantes brasileiros, com escolaridade média no ensino fundamental/médio completo e maior acesso a eletrodomésticos e tecnologia.
- **Níveis VII e VIII**: Os patamares mais elevados da escala, caracterizados por maior escolaridade parental (ensino superior) e maior infraestrutura de bens e serviços contratados no domicílio.

Para fazer essa avaliação, foram adotadas duas abordagens a serem geradas em uma tabela unificada.

1) Foi calculada a média de proficiência por nível socioeconômico para identificar diferenças de desempenho entre os grupos. Como as notas do SAEB são baseadas na TRI, também foram criadas categorias de proficiência para facilitar a interpretação dos resultados.

2) Para aprofundar a análise, foram calculados os percentuais de escolas em cada faixa de proficiência — Abaixo do Básico, Básico, Adequado e Avançado — por nível socioeconômico. As quatro categorias totalizam 100% das escolas de cada grupo, permitindo analisar não apenas a média, mas também como as escolas se distribuem entre as diferentes faixas de desempenho. O mesmo foi realizado para matemática

## 3.1. Tratamento da Base

In [0]:
# Pega automaticamente todas as categorias existentes da coluna PROFICIENCIA_9EF_LP no df_saeb_filtrado (português)
categorias_lp = [
    r["PROFICIENCIA_9EF_LP"]
    for r in df_saeb_filtrado
        .select("PROFICIENCIA_9EF_LP")
        .distinct()
        .collect()
    if r["PROFICIENCIA_9EF_LP"] is not None
]


# Pega automaticamente todas as categorias existentes da coluna PROFICIENCIA_9EF_MT no df_saeb_filtrado (matemática)

categorias_mt = [
    r["PROFICIENCIA_9EF_MT"]
    for r in df_saeb_filtrado
        .select("PROFICIENCIA_9EF_MT")
        .distinct()
        .collect()
    if r["PROFICIENCIA_9EF_MT"] is not None
]

# Monta as agregações básicas de média
agregacoes = [
    F.count("*").alias("quantidade_escolas"),
    F.round(F.avg("MEDIA_9EF_LP"), 2).alias("media_9EF_LP"),
    F.round(F.avg("MEDIA_9EF_MT"), 2).alias("media_9EF_MT")
]

# Geração de Percentuais de cada categoria de LP
for categoria in categorias_lp: #interação para gerar todas as colunas
    nome_coluna = f"{categoria}_LP"  #criação de coluna de LP

    agregacoes.append(   #calculo da percentagem da categoria gerada
        F.round(
            F.sum(
                F.when(
                    F.col("PROFICIENCIA_9EF_LP") == categoria, 1
                ).otherwise(0)
            ) / F.count("*") * 100,
            2
        ).alias(nome_coluna)
    )

# Percentuais de cada categoria de MT
for categoria in categorias_mt:
    nome_coluna = f"{categoria}_MT"

    agregacoes.append(  #calculo da percentagem da categoria gerada
        F.round(
            F.sum(
                F.when(
                    F.col("PROFICIENCIA_9EF_MT") == categoria, 1
                ).otherwise(0)
            ) / F.count("*") * 100,
            2
        ).alias(nome_coluna)
    )

# Agrupamento final  #realização do agrupamento
df_resumo_socioeconomico = (
    df_saeb_filtrado
    .groupBy("NIVEL_SOCIO_ECONOMICO")
    .agg(*agregacoes)
    .orderBy("NIVEL_SOCIO_ECONOMICO")
)

# ------------------------------------------------------------
# Classificação de lingua portuguesa para o 9º ano do ensino fundamental para maior clareza da média do nível
# ------------------------------------------------------------
df_resumo_socioeconomico = df_resumo_socioeconomico.withColumn(
    "PROFICIENCIA_9EF_LP",
    when(col("MEDIA_9EF_LP").isNull(), "Sem_resultado")
    .when(col("MEDIA_9EF_LP") <= 225, "Abaixo do Básico")
    .when(col("MEDIA_9EF_LP") <= 275, "Básico")
    .when(col("MEDIA_9EF_LP") <= 325, "Adequado")
    .otherwise("Avançado")
)


# ------------------------------------------------------------
# Classificação de matemática para o 9º ano do ensino fundamental para maior clareza da média do nível
# ------------------------------------------------------------
df_resumo_socioeconomico = df_resumo_socioeconomico.withColumn(
    "PROFICIENCIA_9EF_MT",
    when(col("MEDIA_9EF_MT").isNull(), "Sem_resultado")
    .when(col("MEDIA_9EF_MT") <= 225, "Abaixo do Básico")
    .when(col("MEDIA_9EF_MT") <= 300, "Básico")
    .when(col("MEDIA_9EF_MT") <= 350, "Adequado")
    .otherwise("Avançado")
)


#reorganização das colunas com o objetivo de facilitar a visualização
df_resumo_socioeconomico = df_resumo_socioeconomico.select(
    # Identificação
    "NIVEL_SOCIO_ECONOMICO",
    "quantidade_escolas",

    # Língua Portuguesa
   "PROFICIENCIA_9EF_LP",
    "media_9EF_LP",
    "Abaixo do Básico_LP",
    "Básico_LP",
    "Adequado_LP",
    "Avançado_LP",

    # Matemática
    "PROFICIENCIA_9EF_MT",
    "media_9EF_MT",
    "Abaixo do Básico_MT",
    "Básico_MT",
    "Adequado_MT",
    "Avançado_MT"
)

#Ajuste de nomenclatura de coluna para salvar no databricks
df_resumo_socioeconomico = df_resumo_socioeconomico.withColumnRenamed("Abaixo do Básico_MT", "Abaixo_do_Básico_MT")
df_resumo_socioeconomico = df_resumo_socioeconomico.withColumnRenamed("Abaixo do Básico_LP", "Abaixo_do_Básico_LP")


display(df_resumo_socioeconomico)

NIVEL_SOCIO_ECONOMICO,quantidade_escolas,PROFICIENCIA_9EF_LP,media_9EF_LP,Abaixo_do_Básico_LP,Básico_LP,Adequado_LP,Avançado_LP,PROFICIENCIA_9EF_MT,media_9EF_MT,Abaixo_do_Básico_MT,Básico_MT,Adequado_MT,Avançado_MT
N�vel I,23,Básico,232.18,39.13,43.48,17.39,0.0,Básico,243.71,43.48,39.13,17.39,0.0
N�vel II,1937,Básico,227.32,51.37,44.24,4.08,0.31,Básico,229.56,52.04,44.45,2.48,1.03
N�vel III,7502,Básico,240.86,22.29,71.99,5.16,0.56,Básico,240.24,25.73,71.37,1.79,1.12
N�vel IV,8568,Básico,249.41,7.74,85.34,6.79,0.13,Básico,246.16,10.45,88.38,0.92,0.26
N�vel V,9952,Básico,259.84,1.34,83.93,14.68,0.05,Básico,257.19,2.09,96.81,1.07,0.03
N�vel VI,2977,Básico,274.38,0.37,50.18,49.41,0.03,Básico,274.33,0.3,92.78,6.89,0.03
N�vel VII,121,Adequado,295.07,0.0,9.09,90.08,0.83,Adequado,303.2,0.0,47.11,52.89,0.0


##3.2. Salvando a base na camada gold

In [0]:
df_resumo_socioeconomico.write.format("delta").mode("overwrite").saveAsTable("resultado_saeb_por_nivel_socioeconomico")


## 3.3. Conclusões:

1) Dos 7 niveis avaliados:
-  **6** apresentam um nível **básico** de proficiência para português e matemática, 
- somente o **nível 7**, com maior escolaridade parental e infraestrutura **apresentou um nível adequado**.

2) **Avaliando as médias**, é possível identificar excluindo o nível 1 que apresenta uma amostra pequena (somente 23 escolas) que:
-  há uma **clara correlação entre maior nível socioeconômico e maior notas do SAEB**

2) Entretanto, Apesar de **6 dos 7 níveis apresentarem nota média**  para português e matemática num nível básico, eles **estão em faixas muito diferentes dessa escala**, com:
-  o nível 2 estando 2 pontos acima do corte inferior do nível básico ( média 227 vs corte 225)
- o **nível 6 estando com a média a menos de 1 ponto de chegar ao nível adequado (média 274 vs corte do nível adequado de 275).**

3) Esse ponto é fortalecido pelas colunas geradas de proficiência, com:
-  o **nível 2** apresentando  **mais da metade das escolas com níveis abaixo do básico para português (51%) e matemática (52%)**
- Esse indice que cai para **menos de 1% no nível 6**

# 4. Geração de tabela por estado para resposta das perguntas 3, 4 e 5

3) Quais estados apresentam os melhores e piores desempenhos médios no SAEB?
4) Quais estados entregam um desempenho educacional abaixo do esperado para seu nível de desenvolvimento?
5) Existem estados que se destacam por apresentar bons resultados educacionais mesmo possuindo indicadores socioeconômicos inferiores?


Para facilitar a resposta a esse tipo de perguntas é necessário gerar uma tabela unificada com os desempenhos médios por estado no SAEB e com os indicadores socioeconômicos por estado. Para isso, os seguintes passos foram realizados:


4.1) Agrupamento dos resultados do saeb por escola para português e matemática

4.2.)Realização de join com a tabela dimensão df_atributos para obter dados gerais dos estados (obs: criada etapa 4.2.1 para avaliar a qualidade desse join, verificando se houve multiplicação de linhas, por exemplo)

4.3.) Realização de join com a tabela dimensão df_indicadores para obter os indicadores socioeconômicos dos estados (obs: criada etapa 4.3.1 para avaliar a qualidade desse join, verificando se houve multiplicação de linhas, por exemplo)

4.4.) Para visualmente facilitar a análise foram criados dois rankings, um pelo indicador de IDHM que foi o escolhido para a análise e outro fazendo o valor médio entre as notas de português e matemática para cada estado. Com isso, a comparação passará a ser feita a partir de posição entre os estados, facilitando a visualização

4.5.) Além disso, foi feita uma etapa para separar esses 2 rankings em quartis e criada uma métrica para avaliar se o estado está no mesmo quartil nos dois rankings ou se está em ranking diferentes, permitindo rapidamente identificar quem está acima ou abaixo do esperado, ajudando a responder as perguntas 4 e 5

## 4.1. Agrupamento dos resultados do SAEB por escola para português e matemática

Calculado a partir da média dessas duas disciplinas

In [0]:
df_saeb_estado = (
    df_saeb_filtrado
    .groupBy("ESTADO")
    .agg(
        F.avg("MEDIA_9EF_LP").alias("MEDIA_9EF_LP"),
        F.avg("MEDIA_9EF_MT").alias("MEDIA_9EF_MT")
    )
)

In [0]:
display(df_saeb_estado.limit(10))

ESTADO,MEDIA_9EF_LP,MEDIA_9EF_MT
Rondônia,246.9945625000001,246.5357187499996
Acre,248.06467153284672,243.12912408759115
Amazonas,240.06508902077147,236.9663501483677
Roraima,221.75977011494254,220.20885057471267
Pará,236.19081050228294,233.48153538812784
Amapá,236.25546218487403,228.67605042016802
Tocantins,242.60007281553408,241.41070388349513
Maranhão,232.0924724986908,228.76839706652711
Piauí,246.47617868675977,246.96570505920317
Ceará,265.6127029914531,267.86457799145273


## 4.2. Join com a tabela df_atributos_estados

Utilizando pela parte da tabela de resultados do saeb a coluna ESTADO como chave e na df_atributos_estados a coluna Estado. Usado left join para não perder registros de estados da tabela saeb em caso de algum erro na df_atributos_estados

In [0]:
# ============================================================
# 4) LEFT JOIN com df_atributos_estados
# ============================================================

df_saeb_atributos = (
    df_saeb_estado.alias("saeb")
    .join(
        df_atributos_estados.alias("atrib"),
        F.col("saeb.ESTADO") == F.col("atrib.Estado"),
        "left"
    )
    .select(
        F.col("saeb.ESTADO"),
        F.col("saeb.MEDIA_9EF_LP"),
        F.col("saeb.MEDIA_9EF_MT"),
        F.col("atrib.Região")
    )
)

In [0]:
display(df_saeb_atributos.limit(10))

ESTADO,MEDIA_9EF_LP,MEDIA_9EF_MT,Região
Sergipe,240.2911061946902,237.14174778761088,Nordeste
Minas Gerais,252.24584100675264,249.873075506446,Sudeste
Pernambuco,248.84409629629633,248.08491111111087,Nordeste
Amapá,236.25546218487403,228.67605042016802,Norte
Rondônia,246.9945625000001,246.5357187499996,Norte
Ceará,265.6127029914531,267.86457799145273,Nordeste
São Paulo,261.9075988100306,259.2013557161069,Sudeste
Espírito Santo,261.92147776183623,261.0104160688666,Sudeste
Acre,248.06467153284672,243.12912408759115,Norte
Santa Catarina,263.78403693931403,264.5467897977134,Sul


###4.2.1. Qualidade da transformação

Foram feitas checagens para verificar se o quantitativo de linhas antes é igual a depois do join. 

In [0]:
# ============================================================
# 5) Checagem do primeiro JOIN
# ============================================================

# Quantidade de linhas antes e depois
qtd_antes_join_1 = df_saeb_estado.count()
qtd_depois_join_1 = df_saeb_atributos.count()

print("=== CHECAGEM JOIN 1 ===")
print(f"Linhas antes do join:  {qtd_antes_join_1}")
print(f"Linhas depois do join: {qtd_depois_join_1}")

if qtd_depois_join_1 > qtd_antes_join_1:
    print("ATENÇÃO: houve multiplicação de linhas no primeiro join.")
else:
    print("OK: não houve multiplicação de linhas.")


# Estados sem correspondência na dimensão
sem_join_1 = (
    df_saeb_atributos
    .filter(F.col("Região").isNull())
    .select("ESTADO")
    .distinct()
)

qtd_sem_join_1 = sem_join_1.count()

print(f"Estados sem correspondência em df_atributos_estados: {qtd_sem_join_1}")

if qtd_sem_join_1 > 0:
    print("Estados sem correspondência:")
    sem_join_1.show(truncate=False)
else:
    print("OK: todos os estados encontraram correspondência.")


# Checagem adicional:
# Verificar se a chave Estado é realmente única na dimensão
duplicados_atributos = (
    df_atributos_estados
    .groupBy("Estado")
    .count()
    .filter(F.col("count") > 1)
)

qtd_duplicados_atributos = duplicados_atributos.count()

print(f"Estados duplicados na dimensão: {qtd_duplicados_atributos}")

if qtd_duplicados_atributos > 0:
    duplicados_atributos.show(truncate=False)

=== CHECAGEM JOIN 1 ===
Linhas antes do join:  27
Linhas depois do join: 27
OK: não houve multiplicação de linhas.
Estados sem correspondência em df_atributos_estados: 0
OK: todos os estados encontraram correspondência.
Estados duplicados na dimensão: 0


##4.3. Join com a tabela de indicadores socioeconômicos

utlizando a coluna ESTADO no dataframe que vinha sendo trabalho (df_saeb_atributos) e UFN na df_indicadores_filtrado

In [0]:
# ============================================================
# 7) LEFT JOIN com df_indicadores_estados
#    df_saeb_atributos.Estado → df_indicadores_filtrado.UFN
# ============================================================

df_final = (
    df_saeb_atributos.alias("saeb")
    .join(
        df_indicadores_filtrado.alias("ind"),
        F.col("saeb.ESTADO") == F.col("ind.UFN"),
        "left"
    )
    .select(
        # Todas as colunas de df_saeb_atributos
        *[F.col(f"saeb.{c}") for c in df_saeb_atributos.columns],

        # Todas as colunas de df_indicadores_filtrado, exceto UFN
        *[
            F.col(f"ind.{c}")
            for c in df_indicadores_filtrado.columns
            if c != "UFN"
        ]
    )
)



In [0]:
display(df_final.limit(30))

ESTADO,MEDIA_9EF_LP,MEDIA_9EF_MT,Região,ANO,E_ANOSESTUDO,GINI,RDPC,IDHM,IDHM_E,IDHM_L,IDHM_R
Sergipe,240.2911061946902,237.14174778761088,Nordeste,2010,9.0100000000000000,0.62000000000000000,523.53000000000000,0.66500000000000004,0.56000000000000005,0.78100000000000003,0.67200000000000004
Minas Gerais,252.24584100675264,249.873075506446,Sudeste,2010,9.3800000000000008,0.56000000000000005,749.69000000000000,0.73099999999999998,0.63800000000000001,0.83799999999999997,0.73000000000000000
Pernambuco,248.84409629629633,248.08491111111087,Nordeste,2010,9.1300000000000008,0.62000000000000000,525.64000000000000,0.67300000000000004,0.57399999999999995,0.78900000000000003,0.67300000000000004
Amapá,236.25546218487403,228.67605042016802,Norte,2010,9.4400000000000000,0.60000000000000000,598.98000000000000,0.70799999999999996,0.62900000000000000,0.81299999999999994,0.69399999999999995
Rondônia,246.9945625000001,246.5357187499996,Norte,2010,9.1999999999999993,0.56000000000000005,670.82000000000000,0.69000000000000000,0.57699999999999996,0.80000000000000000,0.71199999999999997
Ceará,265.6127029914531,267.86457799145273,Nordeste,2010,9.8200000000000000,0.61000000000000000,460.63000000000000,0.68200000000000005,0.61499999999999999,0.79300000000000004,0.65100000000000002
São Paulo,261.9075988100306,259.2013557161069,Sudeste,2010,10.3300000000000000,0.56000000000000005,1084.46000000000000,0.78300000000000003,0.71899999999999997,0.84499999999999997,0.78900000000000003
Espírito Santo,261.92147776183623,261.0104160688666,Sudeste,2010,9.3600000000000000,0.56000000000000005,815.43000000000000,0.74000000000000000,0.65300000000000002,0.83499999999999996,0.74299999999999999
Acre,248.06467153284672,243.12912408759115,Norte,2010,8.6900000000000000,0.63000000000000000,522.15000000000000,0.66300000000000003,0.55900000000000005,0.77700000000000002,0.67100000000000004
Santa Catarina,263.78403693931403,264.5467897977134,Sul,2010,10.2400000000000000,0.49000000000000000,983.90000000000000,0.77400000000000002,0.69699999999999995,0.86000000000000000,0.77300000000000002


### 4.3.1. Qualidade dos dados

Checagens para verificar se houve duplicações/perdas de linhas ou não

In [0]:
# ============================================================
# 8) Checagem do segundo JOIN
# ============================================================

qtd_antes_join_2 = df_saeb_atributos.count()
qtd_depois_join_2 = df_final.count()

print("=== CHECAGEM JOIN 2 ===")
print(f"Linhas antes do join:  {qtd_antes_join_2}")
print(f"Linhas depois do join: {qtd_depois_join_2}")

if qtd_depois_join_2 > qtd_antes_join_2:
    print("ATENÇÃO: houve multiplicação de linhas no segundo join.")
else:
    print("OK: não houve multiplicação de linhas.")


# Verificar ESTADOS sem correspondência em UFN
sem_join_2 = (
    df_saeb_atributos
    .select("ESTADO")
    .distinct()
    .join(
        df_indicadores_filtrado
        .select("UFN")
        .distinct(),
        F.col("ESTADO") == F.col("UFN"),
        "left_anti"
    )
)

qtd_sem_join_2 = sem_join_2.count()

print(
    f"Estados sem correspondência em "
    f"df_indicadores_estados: {qtd_sem_join_2}"
)

if qtd_sem_join_2 > 0:
    print("Estados sem correspondência:")
    sem_join_2.show(truncate=False)
else:
    print("OK: todos os Estados encontraram correspondência.")

=== CHECAGEM JOIN 2 ===
Linhas antes do join:  27
Linhas depois do join: 27
OK: não houve multiplicação de linhas.
Estados sem correspondência em df_indicadores_estados: 0
OK: todos os Estados encontraram correspondência.


## 4.4. Criação dos rankings


Ranking de notas criado a partir de uma nota unificada dada por:  (nota de matemática + nota de português)/2

Ranking socioeconômico criado a partir do indicador IDHM, que foi o selecionado para essa análise, ranqueando do maior IDH até o menor

In [0]:
# ============================================================
# 9) Criação dos rankings
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window



# ------------------------------------------------------------
# Ranking de IDHM
# ------------------------------------------------------------

# Ranking: maior IDHM = maior posição no ranking
window_idhm = Window.orderBy(
    F.col("IDHM").desc()
)

df_final = (
    df_final
    .withColumn(
        "RANKING_IDHM",
        F.dense_rank().over(window_idhm)
    )
)

# ------------------------------------------------------------
# Ranking de desempenho no SAEB - 9º EF
# ------------------------------------------------------------




# Média unificada das proficiências de Língua Portuguesa
# e Matemática
df_final = (
    df_final
    .withColumn(
        "MEDIA_SAEB_9EF",
        (
            F.col("MEDIA_9EF_LP") +
            F.col("MEDIA_9EF_MT")
        ) / 2
    )
)

# Ranking: maior média de proficiência = melhor posição
window_saeb = Window.orderBy(
    F.col("MEDIA_SAEB_9EF").desc()
)
df_final = (
    df_final
    .withColumn(
        "RANKING_SAEB_9EF",
        F.dense_rank().over(window_saeb)
    )
)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(df_final.limit(30))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


ESTADO,MEDIA_9EF_LP,MEDIA_9EF_MT,Região,ANO,E_ANOSESTUDO,GINI,RDPC,IDHM,IDHM_E,IDHM_L,IDHM_R,RANKING_IDHM,MEDIA_SAEB_9EF,RANKING_SAEB_9EF
Ceará,265.6127029914531,267.86457799145273,Nordeste,2010,9.8200000000000000,0.61000000000000000,460.63000000000000,0.68200000000000005,0.61499999999999999,0.79300000000000004,0.65100000000000002,17,266.7386404914529,1
Paraná,264.0033572281962,265.40866786141,Sul,2010,10.4300000000000000,0.53000000000000000,890.89000000000000,0.74900000000000000,0.66800000000000004,0.83000000000000000,0.75700000000000001,5,264.7060125448031,2
Goiás,264.44657286432135,263.9559698492461,Centro-Oeste,2010,9.7200000000000006,0.55000000000000004,810.97000000000000,0.73499999999999999,0.64600000000000002,0.82699999999999996,0.74199999999999999,8,264.20127135678376,3
Santa Catarina,263.78403693931403,264.5467897977134,Sul,2010,10.2400000000000000,0.49000000000000000,983.90000000000000,0.77400000000000002,0.69699999999999995,0.86000000000000000,0.77300000000000002,3,264.1654133685137,4
Rio Grande do Sul,263.41532127558304,260.81853879105165,Sul,2010,10.0000000000000000,0.54000000000000000,959.24000000000000,0.74600000000000000,0.64200000000000002,0.84000000000000000,0.76900000000000002,6,262.1169300333173,5
Espírito Santo,261.92147776183623,261.0104160688666,Sudeste,2010,9.3600000000000000,0.56000000000000005,815.43000000000000,0.74000000000000000,0.65300000000000002,0.83499999999999996,0.74299999999999999,7,261.4659469153514,6
São Paulo,261.9075988100306,259.2013557161069,Sudeste,2010,10.3300000000000000,0.56000000000000005,1084.46000000000000,0.78300000000000003,0.71899999999999997,0.84499999999999997,0.78900000000000003,2,260.55447726306875,7
Alagoas,248.78934740882912,254.16226487524017,Nordeste,2010,9.0700000000000000,0.63000000000000000,432.56000000000000,0.63100000000000001,0.52000000000000000,0.75500000000000000,0.64100000000000001,26,251.47580614203463,8
Rio de Janeiro,253.51941134242665,248.84054558506824,Sudeste,2010,9.1700000000000000,0.59000000000000000,1039.30000000000000,0.76100000000000001,0.67500000000000004,0.83499999999999996,0.78200000000000003,4,251.17997846374743,9
Minas Gerais,252.24584100675264,249.873075506446,Sudeste,2010,9.3800000000000008,0.56000000000000005,749.69000000000000,0.73099999999999998,0.63800000000000001,0.83799999999999997,0.73000000000000000,9,251.05945825659933,10


## 4.5. Criação dos quartis e métrica CLASSIFICACAO_IDHM_SAEB

Os quartis foram gerados para o ranking de notas e para o de IDHM. E

Para os quartis tanto de notas quanto de IDHM, quanto menor o valor, melhor o resultado, logo:

 Quartil 1 = estados com maiores médias no SAEB

 Quartil 4 = estados com menores médias no SAEB


Em seguida, foi criada a métrica CLASSIFICACAO_IDHM_SAEB que fez a seguinte categorização:

1) Caso os quartis sejam iguais: Em linha com o esperado
2) Caso o quartil de nota seja 1 acima do quartil de IDHM: Levemente acima do esperado
3) Caso o quartil de nota seja 2 acima do quartil de IDHM: Acima do esperado
4) Caso o quartil de nota seja 3 acima do quartil de IDHM: Muito acima do esperado

2) Caso o quartil de nota seja 1 abaixo do quartil de IDHM: Levemente abaixo do esperado
3) Caso o quartil de nota seja 2 abaixo do quartil de IDHM: Abaixo do esperado
4) Caso o quartil de nota seja 3 abaixo do quartil de IDHM: Muito abaixo do esperado

In [0]:
# ============================================================
# 10) CRIAÇÃO DOS QUARTIS
# ============================================================

# ------------------------------------------------------------
# 10.1) Quartil do SAEB
# ------------------------------------------------------------
# Quartil 1 = estados com maiores médias no SAEB
# Quartil 4 = estados com menores médias no SAEB

window_quartil_saeb = Window.orderBy(
    F.col("MEDIA_SAEB_9EF").desc()
)

df_final = (
    df_final
    .withColumn(
        "QUARTIL_SAEB",
        F.ntile(4).over(window_quartil_saeb)
    )
)


# ------------------------------------------------------------
# 10.2) Quartil do IDHM
# ------------------------------------------------------------
# Quartil 1 = estados com maiores IDHM
# Quartil 4 = estados com menores IDHM

window_quartil_idhm = Window.orderBy(
    F.col("IDHM").desc()
)

df_final = (
    df_final
    .withColumn(
        "QUARTIL_IDHM",
        F.ntile(4).over(window_quartil_idhm)
    )
)


# ============================================================
# 11) DIFERENÇA ENTRE OS QUARTIS
# ============================================================

df_final = (
    df_final
    .withColumn(
        "DIF_QUARTIS",
        F.abs(
            F.col("QUARTIL_SAEB") -
            F.col("QUARTIL_IDHM")
        )
    )
)


# ============================================================
# 12) CLASSIFICAÇÃO DA RELAÇÃO IDHM x SAEB
# ============================================================
#
# Quartil SAEB menor que quartil IDHM:
# → posição relativa do SAEB é melhor
#
# Quartil SAEB maior que quartil IDHM:
# → posição relativa do SAEB é pior
#
# Mesmos quartis:
# → resultados em linha
# ============================================================

df_final = (
    df_final
    .withColumn(
        "CLASSIFICACAO_IDHM_SAEB",

        # Mesmo quartil
        F.when(
            F.col("DIF_QUARTIS") == 0,
            "Em linha com o esperado"
        )

        # SAEB 1 quartil acima
        .when(
            (F.col("QUARTIL_SAEB") < F.col("QUARTIL_IDHM")) &
            (F.col("DIF_QUARTIS") == 1),
            "Levemente acima do esperado"
        )

        # SAEB 2 quartis acima
        .when(
            (F.col("QUARTIL_SAEB") < F.col("QUARTIL_IDHM")) &
            (F.col("DIF_QUARTIS") == 2),
            "Acima do esperado"
        )

        # SAEB 3 quartis acima
        .when(
            (F.col("QUARTIL_SAEB") < F.col("QUARTIL_IDHM")) &
            (F.col("DIF_QUARTIS") == 3),
            "Muito acima do esperado"
        )

        # SAEB 1 quartil abaixo
        .when(
            (F.col("QUARTIL_SAEB") > F.col("QUARTIL_IDHM")) &
            (F.col("DIF_QUARTIS") == 1),
            "Levemente abaixo do esperado"
        )

        # SAEB 2 quartis abaixo
        .when(
            (F.col("QUARTIL_SAEB") > F.col("QUARTIL_IDHM")) &
            (F.col("DIF_QUARTIS") == 2),
            "Abaixo do esperado"
        )

        # SAEB 3 quartis abaixo
        .when(
            (F.col("QUARTIL_SAEB") > F.col("QUARTIL_IDHM")) &
            (F.col("DIF_QUARTIS") == 3),
            "Muito abaixo do esperado"
        )
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


### 4.5.1. Salvando a base na camada gold

Base salva na camada gold pronta para análises. Mantidos demais indicadores

In [0]:
# seleção das colunas de interesse para a análise de IDHM

df_final = (df_final.select(
    "ESTADO",
    "MEDIA_9EF_LP",
    "MEDIA_9EF_MT",
    "MEDIA_SAEB_9EF",
    "IDHM",
    "RANKING_SAEB_9EF",
    "RANKING_IDHM",
    "QUARTIL_SAEB",
    "QUARTIL_IDHM",
    "DIF_QUARTIS",
    "CLASSIFICACAO_IDHM_SAEB"
).orderBy(
    "DIF_QUARTIS",
    F.col("MEDIA_SAEB_9EF").desc()))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
df_final.write.format("delta").mode("overwrite").saveAsTable("resultado_saeb_por_estado_vs_idhm")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## 4.6. Visualização e Análise

In [0]:
# ============================================================
# 13) VISUALIZAÇÃO DO RESULTADO
# ============================================================

display(df_final.select(
    "ESTADO",
    "MEDIA_9EF_LP",
    "MEDIA_9EF_MT",
    "MEDIA_SAEB_9EF",
    "IDHM",
    "RANKING_SAEB_9EF",
    "RANKING_IDHM",
    "QUARTIL_SAEB",
    "QUARTIL_IDHM",
    "DIF_QUARTIS",
    "CLASSIFICACAO_IDHM_SAEB"
).orderBy(
    "DIF_QUARTIS",
    F.col("MEDIA_SAEB_9EF").desc())
.limit(30))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


ESTADO,MEDIA_9EF_LP,MEDIA_9EF_MT,MEDIA_SAEB_9EF,IDHM,RANKING_SAEB_9EF,RANKING_IDHM,QUARTIL_SAEB,QUARTIL_IDHM,DIF_QUARTIS,CLASSIFICACAO_IDHM_SAEB
Paraná,264.0033572281962,265.40866786141,264.7060125448031,0.74900000000000000,2,5,1,1,0,Em linha com o esperado
Santa Catarina,263.78403693931403,264.5467897977134,264.1654133685137,0.77400000000000002,4,3,1,1,0,Em linha com o esperado
Rio Grande do Sul,263.41532127558304,260.81853879105165,262.1169300333173,0.74600000000000000,5,6,1,1,0,Em linha com o esperado
Espírito Santo,261.92147776183623,261.0104160688666,261.4659469153514,0.74000000000000000,6,7,1,1,0,Em linha com o esperado
São Paulo,261.9075988100306,259.2013557161069,260.55447726306875,0.78300000000000003,7,2,1,1,0,Em linha com o esperado
Minas Gerais,252.24584100675264,249.873075506446,251.05945825659933,0.73099999999999998,10,9,2,2,0,Em linha com o esperado
Mato Grosso do Sul,251.94486111111107,247.36452777777768,249.65469444444437,0.72899999999999998,12,10,2,2,0,Em linha com o esperado
Acre,248.06467153284672,243.12912408759115,245.59689781021893,0.66300000000000003,16,21,3,3,0,Em linha com o esperado
Sergipe,240.2911061946902,237.14174778761088,238.71642699115054,0.66500000000000004,20,20,3,3,0,Em linha com o esperado
Amazonas,240.06508902077147,236.9663501483677,238.51571958456958,0.67400000000000004,21,18,3,3,0,Em linha com o esperado


In [0]:
# Cálculo da correlação entre IDHM e a nota média do SAEB
corr = df_final.stat.corr("IDHM", "MEDIA_SAEB_9EF", method="pearson")
print(f"Correlação: {corr}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Correlação: 0.5231801999706021


**3) Quais estados apresentam os melhores e piores desempenhos médios no SAEB?**

A partir da tabela gerada acima, os estados com melhores desempenhos são Ceará, Paraná, Goiás e Santa Catarina

**4) Quais estados entregam um desempenho educacional abaixo do esperado para seu nível de desenvolvimento?**

Tem dois estados que apresentam resultados abaixo do esperado:
- O Amapá, que está em 12º no ranking de IDHM, mas na 25º posição no SAEB
- Roraima, em 13º no ranking de IDHM e na última posição do SAEB

**5) Existem estados que se destacam por apresentar bons resultados educacionais mesmo possuindo indicadores socioeconômicos inferiores?**

Sim, os destaques são:
- O estado do Ceará, com primeiro lugar no ranking do SAEB e 17º lugar no ranking do IDHM
- O estado de Alagoas, com o 8º lugar no ranking do SAEB e 26 no de IDHM


OBS: Foi também calculada a correlação entre IDHM e a nota do SAEB, encontrando um valor de 0.52, que é uma correlação linear positiva moderada.
